# MOVER cohort construction

Audits patient linkage, decodes archive members, and selects window-eligible surgeries. Raw archives and the restricted patient crosswalk remain local.

Run the setup cell first. Full processing and training require locally provisioned datasets; the grading quick start uses synthetic tests and saved aggregate results.


In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
import importlib
from hemodynamics import canonical, mover_signals, windows, mover_pipeline as mover, mover_fast
for _module in (canonical, mover_signals, windows, mover, mover_fast):   # pick up edited code in a long-lived kernel (dependency order)
    importlib.reload(_module)
from hemodynamics.canonical import load_case, ensure_disk_space
import psutil
CONFIG = mover.load_config(ROOT)
MODE = 'FAST'   # FAST = one-pass parallel ingest (recommended) | legacy multi-pass: DISCOVERY / PILOT / FULL_PROCESS / RESUME
VITALDB_DOWNLOAD_RUNNING = False   # True if the VitalDB notebook downloads at the same time: MOVER then uses half the workers
# legacy multi-pass switches (ignored by FAST)
RUN_DISCOVERY = False
COMPLETE_DISCOVERY = True
CREATE_SELECTION = False
RESUME_TARGET = 'DISCOVERY'
FAST = mover_fast._fast(CONFIG)
LADDER, TARGETS = windows.load_ladder(CONFIG)
vm = psutil.virtual_memory()
print(json.dumps({
    'project': str(ROOT), 'mode': MODE, 'logical_cpus': os.cpu_count(),
    'eligibility_mode': CONFIG.get('eligibility_mode'), 'ingest_profile': mover_fast._profile(CONFIG),
    'decode_workers': mover_fast._resolve_workers(FAST, None, VITALDB_DOWNLOAD_RUNNING, 'decode_workers'),
    'finalize_workers': mover_fast._resolve_workers(FAST, None, VITALDB_DOWNLOAD_RUNNING, 'finalize_workers'),
    'archive_readers': FAST['archive_readers'], 'payload_queue_MB': FAST['max_inflight_payload_mb'],
    'keep_free_RAM_GB': FAST['min_available_ram_gb'], 'RAM_now_free_GB': round(vm.available / 1024**3, 1), 'RAM_total_GB': round(vm.total / 1024**3, 1),
    'disk_free_GB': round(ensure_disk_space(ROOT, int(CONFIG['min_free_space_gb'] * 1e9)) / 1e9, 1), 'disk_reserve_GB': CONFIG['min_free_space_gb'],
    'staging_cap_GB': FAST['staging_cap_gb'], 'ladder': LADDER, 'targets': TARGETS, 'window_rule': windows.load_rule(CONFIG)}, indent=2))
if (ROOT / 'data' / 'checkpoints' / 'mover' / 'fast_ingest.lock').exists():
    print('WARNING: a MOVER run lock exists. If no run is active it is stale and will be taken over; if one is active, do not start a second.')
print('SETUP ONLY - nothing is running yet. Run the cell under "2. RUN PROCESSING" next. Stop a running cell safely with the notebook Interrupt button.', flush=True)

## 1. Patient identity and surgery timing
PAT_ID is patient identity; LOG_ID is surgery. Filename patient/time is matched to verified local OR windows converted to UTC. Four ambiguous surgery linkages are excluded. DST-ambiguous clinical timestamps are excluded. No patient-independent split is claimed until the final patient selection.

In [ ]:
audit_path = ROOT / 'data' / 'manifests' / 'mover_patient_linkage_audit.json'
if not audit_path.exists():
    mover.audit_linkage(ROOT)
print(json.dumps(json.loads(audit_path.read_text()), indent=2))

## 2. Cohort processing

Run the configuration cell above, then the code cell directly below. Do not run it while the VitalDB notebook is mid-download unless you set `VITALDB_DOWNLOAD_RUNNING = True` (MOVER then uses half the workers).

**FAST mode** shows one compressed-byte bar per archive plus an overall bar (members/PLETH members/decoded/fragments/failures, queue MB, free RAM, free disk, staging GB, canary status). Then it assesses every candidate surgery window-by-window (progress bar `assess surgeries`), applies the eligibility ladder, writes the cohort (`write cases`) and ends with one banner: **COMPLETE / PAUSED / INCOMPLETE / FAILED**. Archive-read 100% is not completion. On COMPLETE the banner lists the ladder funnel, the chosen rung, ready hours and the patient split counts.

**Safe stop:** press **Interrupt** once. In-flight members finish, checkpoints are saved and the cell prints PAUSED. Rerun the same cell to resume (finished members are skipped; each gzip replays its compressed prefix, which is fast compared with decoding). Interrupt twice to abandon in-flight members. Never run two copies at once (a lock refuses it).

The legacy `DISCOVERY` / `PILOT` / `FULL_PROCESS` modes are kept below but are the slow multi-pass design.

In [ ]:
import importlib
import pandas as pd

# Reload edited helpers and the saved configuration instead of using stale kernel state.
for _module in (canonical, mover_signals, windows, mover, mover_fast):
    importlib.reload(_module)
CONFIG = mover.load_config(ROOT)
effective_mode = RESUME_TARGET if MODE == 'RESUME' else MODE
inventory_path = ROOT / 'data' / 'manifests' / 'mover_archive_inventory.parquet'
print(f'RUN PROCESSING: {effective_mode}', flush=True)

if effective_mode == 'FAST':
    result = mover_fast.run(ROOT, concurrent_vitaldb=VITALDB_DOWNLOAD_RUNNING, progress=True)
    print(json.dumps({k: v for k, v in result.items() if k not in ('ingest', 'eda', 'raw_archive_readiness')}, indent=2, default=str))
    if result['status'] != 'COMPLETE':
        print(f"NOT COMPLETE ({result['status']}). Raw archives are untouched; fix/inspect, then rerun this cell to resume.")
elif effective_mode == 'DISCOVERY':
    if RUN_DISCOVERY or not inventory_path.exists():
        print('Starting archive scan. Live progress appears below; leave this cell running.', flush=True)
        inventory, metrics = mover.discover(ROOT, complete_scan=COMPLETE_DISCOVERY, progress=True)
        display(inventory)
        print(json.dumps(metrics, indent=2))
        if COMPLETE_DISCOVERY and not metrics['full_scan_complete']:
            raise mover.DiscoveryIncomplete('The full scan did not complete. Check archive statuses/error logs before running PILOT.')
    else:
        print('NO SCAN STARTED: displaying cached inventory. Set RUN_DISCOVERY=True in the configuration cell to scan.', flush=True)
        display(pd.read_parquet(inventory_path))
elif effective_mode == 'PILOT':
    inventory = pd.read_parquet(inventory_path) if inventory_path.exists() else pd.DataFrame()
    if len(inventory) != len(CONFIG['mover_archives']) or not inventory.scan_completed.all():
        raise mover.DiscoveryIncomplete('PILOT needs complete archive discovery first. Run the configuration cell with MODE=DISCOVERY, RUN_DISCOVERY=True, COMPLETE_DISCOVERY=True, then run this cell. Cached diagnostic fragments are not an eligible full-case pilot.')
    print('PILOT STARTED: retrieving and validating eligible full cases. Archive retrieval may take hours.', flush=True)
    pilot_metrics = mover.pilot(ROOT)
    print(json.dumps(pilot_metrics, indent=2))
    print('PILOT FINISHED: eligible size estimate valid =', pilot_metrics['full_case_size_estimate_valid'], flush=True)
elif effective_mode == 'FULL_PROCESS':
    if not CONFIG['allow_full_process']:
        raise mover.DiscoveryIncomplete('FULL_PROCESS is disabled. Validate discovery and the eligible full-case pilot first.')
    print('FULL PROCESSING STARTED. Keep raw archives; verify failures and deletion readiness afterward.', flush=True)
    if not (ROOT / 'data' / 'manifests' / 'mover_selected_cohort.parquet').exists():
        display(mover.select_cohort(ROOT))
    print(mover.process_selected(ROOT))
    print('FULL_PROCESS FINISHED: inspect processing failures and readiness reports before considering deletion.', flush=True)
else:
    raise ValueError('Unsupported MODE/RESUME_TARGET')

## 3. Cohort results
FAST mode: the finalize summary (including the eligibility ladder funnel and chosen rung), the frozen patient-level split record, the per-case window statistics (kept for every assessed surgery) and the selected cohort. No selection uses hypotension outcomes.

In [ ]:
import pandas as pd
final_path = ROOT / 'results/mover_fast_finalize.json'
if MODE == 'FAST':
    if final_path.exists():
        finalize = json.loads(final_path.read_text())
        brief = {k: finalize.get(k) for k in ('status', 'selected_cases', 'selected_patients', 'selection', 'at', 'seconds', 'eligible_cases', 'eligible_patients')}
        print(json.dumps({**brief, 'window_eligibility': finalize.get('window_eligibility')}, indent=2, default=str)[:8000])
    else:
        print('Fast ingest not finalized yet - run the cell under 2. RUN PROCESSING.')
    frozen_path = ROOT / 'data/manifests/mover_cohort_frozen.json'
    if frozen_path.exists():
        print('FROZEN COHORT RECORD:', json.dumps(json.loads(frozen_path.read_text()), indent=2)[:3000])
    stats_path = ROOT / 'data/manifests/mover_case_window_stats.parquet'
    if stats_path.exists():
        stats = pd.read_parquet(stats_path)
        if 'rung_passed' in stats:
            display(stats.rung_passed.value_counts().rename('assessed surgeries by strictest rung passed').to_frame())
else:
    metrics_path = ROOT / 'results/mover_pilot_metrics.json'
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2) if metrics_path.exists() else 'Pilot not yet run')
    if CREATE_SELECTION:
        display(mover.select_cohort(ROOT))
selected_path = ROOT / 'data/manifests/mover_selected_cohort.parquet'
if selected_path.exists():
    display(pd.read_parquet(selected_path))
else:
    print('No selected cohort / patient splits yet.')

## 4. EDA and raw-archive deletion readiness
FAST EDA reads manifests/checkpoints only (never the archives). An archive is never safe merely because a loop stopped; readiness needs a gzip-verified pass, verified selected outputs, zero unreadable PLETH members, a validated split and a working loader. **Do not delete the raw archives**: deleting discards unselected surgeries and ART/ECG-only members, and a different eligibility rule would need them.

In [ ]:
if MODE == 'FAST':
    print(json.dumps(mover_fast.fast_eda(ROOT), indent=2, default=str)[:8000])
    display(mover_fast.fast_deletion_readiness(ROOT))
else:
    print(mover.eda(ROOT))
    display(mover.deletion_readiness(ROOT))
    print('Schema audit:', mover.write_schema_audit(ROOT))

## 5. Cross-dataset loader and unified index
Use load_case(ROOT, dataset, case_id) for committed case outputs. It returns ART, ECG, PPG, native MAP, their time axes, patient/split and sampling metadata, and refuses corrupt/incomplete output. The next cell validates both datasets, then builds `data/manifests/unified_case_index.parquet` (one row per completed case of MOVER and VitalDB, with within-dataset and pooled patient-level splits, ready-window statistics and parquet paths). It is incremental: rerun it after VitalDB finishes. Final ML/BSHI readiness additionally requires all selected outputs and zero patient overlap, not merely successful sample loading.

In [ ]:
from hemodynamics.validate_data import validate_outputs
validation = validate_outputs(ROOT)
print(json.dumps(validation, indent=2))

In [ ]:
from hemodynamics.unified import build_unified_index
unified_index = build_unified_index(ROOT)   # indexes any dataset whose ready-window statistics are missing, then joins MOVER + VitalDB
display(unified_index.groupby(['dataset', 'split_within_dataset']).agg(cases=('case_id', 'size'), patients=('patient_key', 'nunique')))